# Round38 Targeted Repair

Notebook para correr a Round38 sem ter de executar comandos manualmente.

Modos disponíveis:

- `smoke_9338`: teste curto para confirmar que tudo funciona.
- `focused_9338`: corrida principal focada no `9338.png`.
- `full_priority`: corre os quatro alvos Round38: `9338.png`, `7836.png`, `1159_3.png`, `1159_7.png`.

Por defeito está em `focused_9338`.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "search_multiseed_validate.py").exists():
    PROJECT_ROOT = Path(r"C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2")

SRC_DIR = PROJECT_ROOT / "src"
PROMPT_BANK = PROJECT_ROOT / "prompts" / "refinement_round38_targeted_repair.json"
TARGETS_DIR = PROJECT_ROOT / "TP2-students" / "students" / "tp2-chosen"
OUTPUT_DIR = PROJECT_ROOT / "TP2-students" / "students" / "outputs"

PYTHON_CANDIDATES = [
    PROJECT_ROOT / ".venv_win" / "Scripts" / "python.exe",
    Path(r"C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe"),
    Path(sys.executable),
]

def has_module(python_exe, module_name):
    if not Path(python_exe).exists():
        return False
    result = subprocess.run(
        [str(python_exe), "-c", f"import {module_name}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    return result.returncode == 0

PYTHON_EXE = None
for candidate in PYTHON_CANDIDATES:
    if has_module(candidate, "diffusers"):
        PYTHON_EXE = candidate
        break

if PYTHON_EXE is None:
    raise RuntimeError(
        "No Python with diffusers found. Create a venv with: "
        "python -m venv .venv_win ; .\\.venv_win\\Scripts\\python.exe -m pip install -r .\\requirements.txt"
    )

print("Project root:", PROJECT_ROOT)
print("Notebook kernel:", sys.executable)
print("Render/search Python:", PYTHON_EXE)
print("Prompt bank:", PROMPT_BANK)
print("Targets dir:", TARGETS_DIR)
print("Output dir:", OUTPUT_DIR)

assert (SRC_DIR / "generate_round38_targeted_repair.py").exists(), "Round38 generator not found"
assert (SRC_DIR / "search_multiseed_validate.py").exists(), "Search script not found"
assert TARGETS_DIR.exists(), "Target image folder not found"

Project root: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2
Notebook kernel: c:\Users\tugap\AppData\Local\Programs\Python\Python311\python.exe
Render/search Python: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe
Prompt bank: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round38_targeted_repair.json
Targets dir: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\tp2-chosen
Output dir: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs


## Configuração

Muda só `RUN_MODE` se quiseres trocar entre teste curto, corrida focada ou corrida completa.

In [2]:
RUN_MODE = "focused_9338"  # smoke_9338 | focused_9338 | full_priority

OFFLINE = True
DISABLE_PROGRESS_BAR = True
MAXSTACK_SCORING = True

MODES = {
    "smoke_9338": {
        "identity": "round38_smoke_9338",
        "only": "9338.png",
        "limit_per_target": 20,
        "top_k": 4,
        "stage1_save_k": 8,
        "validation_top_n": 6,
        "ensemble_per_metric": 2,
        "seed_offsets": [1],
    },
    "focused_9338": {
        "identity": "round38_9338_targeted",
        "only": "9338.png",
        "limit_per_target": None,
        "top_k": 12,
        "stage1_save_k": 30,
        "validation_top_n": 20,
        "ensemble_per_metric": 5,
        "seed_offsets": [1, 2, 3],
    },
    "full_priority": {
        "identity": "round38_targeted_repair",
        "only": None,
        "limit_per_target": None,
        "top_k": 12,
        "stage1_save_k": 30,
        "validation_top_n": 20,
        "ensemble_per_metric": 5,
        "seed_offsets": [1, 2, 3],
    },
}

assert RUN_MODE in MODES, f"Invalid RUN_MODE: {RUN_MODE}"
config = MODES[RUN_MODE]
print("Selected mode:", RUN_MODE)
print(json.dumps(config, indent=2))

Selected mode: focused_9338
{
  "identity": "round38_9338_targeted",
  "only": "9338.png",
  "limit_per_target": null,
  "top_k": 12,
  "stage1_save_k": 30,
  "validation_top_n": 20,
  "ensemble_per_metric": 5,
  "seed_offsets": [
    1,
    2,
    3
  ]
}


## Gerar banco de prompts Round38

In [3]:
cmd = [
    str(PYTHON_EXE),
    str(SRC_DIR / "generate_round38_targeted_repair.py"),
    "--output",
    str(PROMPT_BANK),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

data = json.loads(PROMPT_BANK.read_text(encoding="utf-8"))
print({target: len(entries) for target, entries in data.items()})
print("Total prompts:", sum(len(entries) for entries in data.values()))

Running: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\generate_round38_targeted_repair.py --output c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round38_targeted_repair.json
{'9338.png': 560, '7836.png': 360, '1159_3.png': 260, '1159_7.png': 220}
Total prompts: 1400


## Correr pesquisa e validação multi-seed

In [4]:
args = [
    str(PYTHON_EXE),
    str(SRC_DIR / "search_multiseed_validate.py"),
    "--prompts", str(PROMPT_BANK),
    "--targets", str(TARGETS_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--identity", config["identity"],
    "--top-k", str(config["top_k"]),
    "--stage1-save-k", str(config["stage1_save_k"]),
    "--validation-top-n", str(config["validation_top_n"]),
    "--ensemble-per-metric", str(config["ensemble_per_metric"]),
    "--seed-offsets", *[str(seed) for seed in config["seed_offsets"]],
]

if config["only"]:
    args.extend(["--only", config["only"]])
if config["limit_per_target"] is not None:
    args.extend(["--limit-per-target", str(config["limit_per_target"])])
if OFFLINE:
    args.append("--offline")
if DISABLE_PROGRESS_BAR:
    args.append("--disable-progress-bar")
if MAXSTACK_SCORING:
    args.append("--maxstack-scoring")

print("Running:")
print(" ".join(args))

process = subprocess.Popen(
    args,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Search failed with exit code {return_code}")

print("Finished successfully")

Running:
C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\search_multiseed_validate.py --prompts c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round38_targeted_repair.json --targets c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\tp2-chosen --output-dir c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs --identity round38_9338_targeted --top-k 12 --stage1-save-k 30 --validation-top-n 20 --ensemble-per-metric 5 --seed-offsets 1 2 3 --only 9338.png --offline --disable-progress-bar --maxstack-scoring
Couldn't connect to the Hub: Cannot reach https://huggingface.co/api/models/SimianLuo/LCM_Dreamshaper_v7: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable..
Will try to load from loc

## Localizar resultados

In [5]:
run_dirs = sorted(
    [path for path in OUTPUT_DIR.glob(f"*_{config['identity']}") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not run_dirs:
    print("No run directory found yet for identity:", config["identity"])
else:
    latest = run_dirs[0]
    print("Latest run:", latest)
    for item in sorted(latest.glob("*")):
        print(item.name)

Latest run: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs\20260602-211211_round38_9338_targeted
9338
9338_robust_prompt_ranking.csv
9338_selected_for_multiseed.csv
9338_stage1_fixed_seed_top30.csv
contact_sheet_top12_robust.jpg
stage1_fixed_seed_metrics.csv
stage2_aux_seed_metrics.csv
summary.json
top12_robust_fixed_seed.csv
